In [1]:
mssparkutils.fs.ls("Files/bronze/raw")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 3, Finished, Available, Finished, False)

[FileInfo(path=abfss://cd5b76c2-56c2-43b2-88b9-e0e2fa17a964@onelake.dfs.fabric.microsoft.com/1fd97920-eac6-4ef6-8977-9078b3af251d/Files/bronze/raw/olist_customers_dataset.csv, name=olist_customers_dataset.csv, size=9033957),
 FileInfo(path=abfss://cd5b76c2-56c2-43b2-88b9-e0e2fa17a964@onelake.dfs.fabric.microsoft.com/1fd97920-eac6-4ef6-8977-9078b3af251d/Files/bronze/raw/olist_geolocation_dataset.csv, name=olist_geolocation_dataset.csv, size=61273883),
 FileInfo(path=abfss://cd5b76c2-56c2-43b2-88b9-e0e2fa17a964@onelake.dfs.fabric.microsoft.com/1fd97920-eac6-4ef6-8977-9078b3af251d/Files/bronze/raw/olist_order_items_dataset.csv, name=olist_order_items_dataset.csv, size=15438671),
 FileInfo(path=abfss://cd5b76c2-56c2-43b2-88b9-e0e2fa17a964@onelake.dfs.fabric.microsoft.com/1fd97920-eac6-4ef6-8977-9078b3af251d/Files/bronze/raw/olist_order_payments_dataset.csv, name=olist_order_payments_dataset.csv, size=5777138),
 FileInfo(path=abfss://cd5b76c2-56c2-43b2-88b9-e0e2fa17a964@onelake.dfs.fabric.m

In [2]:
files = mssparkutils.fs.ls("Files/bronze/raw")
sample_path = files[0].path
BASE_PATH = sample_path.split("/Files/")[0]
BRONZE_PATH = f"{BASE_PATH}/Files/bronze/raw"

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 4, Finished, Available, Finished, False)

In [3]:
df_orders = spark.read.csv(f"{BRONZE_PATH}/olist_orders_dataset.csv", header=True, inferSchema=True)

df_order_items = spark.read.csv(f"{BRONZE_PATH}/olist_order_items_dataset.csv",header=True, inferSchema=True)

df_customers = spark.read.csv(f"{BRONZE_PATH}/olist_customers_dataset.csv", header=True, inferSchema=True)

df_products = spark.read.csv(f"{BRONZE_PATH}/olist_products_dataset.csv", header=True, inferSchema=True)

df_sellers = spark.read.csv(f"{BRONZE_PATH}/olist_sellers_dataset.csv", header=True, inferSchema=True)

df_payments = spark.read.csv(f"{BRONZE_PATH}/olist_order_payments_dataset.csv", header=True, inferSchema=True)

df_reviews = spark.read.csv(f"{BRONZE_PATH}/olist_order_reviews_dataset.csv", header=True, inferSchema=True)

df_geolocation = spark.read.csv(f"{BRONZE_PATH}/olist_geolocation_dataset.csv", header=True, inferSchema=True)

df_category = spark.read.csv(f"{BRONZE_PATH}/product_category_name_translation.csv", header=True, inferSchema=True)

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 5, Finished, Available, Finished, False)

In [4]:
from pyspark.sql.functions import col, when, current_timestamp, lit

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 6, Finished, Available, Finished, False)

In [5]:
df_orders_silver = (
    df_orders
    .withColumn("delivery_status", when(col("order_delivered_customer_date")
             .isNull(), "NOT_DELIVERED")
        .otherwise("DELIVERED"))
    .withColumn("approval_status",when(col("order_approved_at")
            .isNull(), "NOT_APPROVED")
        .otherwise("APPROVED"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("layer", lit("silver")))
df_orders_silver.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 7, Finished, Available, Finished, False)

Changing first letter capital in custumer_city column in customers table

In [6]:
from pyspark.sql.functions import initcap
df_customers_silver = (
    df_customers.withColumn("customer_city", initcap(col("customer_city")))
)
df_customers_silver.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 8, Finished, Available, Finished, False)

products table needs change is column name, fillna for uncategorized products, joining with category table for category to get english names from category table

In [7]:
df_products_silver = (
    df_products
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_category_name", 
                when(col("product_category_name").isNull(), "uncategorized")
                .otherwise(col("product_category_name")))
    .withColumn("product_weight_g",
                when(col("product_weight_g").isNull(), 0)
                .otherwise(col("product_weight_g")))
    .withColumn("product_length_cm",
                when(col("product_length_cm").isNull(), 0)
                .otherwise(col("product_length_cm")))
    .withColumn("product_height_cm",
                when(col("product_height_cm").isNull(), 0)
                .otherwise(col("product_height_cm")))
    .withColumn("product_width_cm",
                when(col("product_width_cm").isNull(), 0)
                .otherwise(col("product_width_cm")))
)

df_products_silver = df_products_silver.join(df_category, on="product_category_name", how="left")
df_products_silver.select("product_id", "product_category_name", "product_category_name_english")
df_products_silver.write.format("delta").mode("overwrite").saveAsTable("silver_products")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 9, Finished, Available, Finished, False)

converting seller city column into captial first letters using initcap

In [8]:
df_sellers_silver = (
    df_sellers
    .withColumn("seller_city", initcap(col("seller_city")))
)
df_sellers_silver.write.format("delta").mode("overwrite").saveAsTable("silver_sellers")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 10, Finished, Available, Finished, False)

converting payment type column into initcap

In [9]:
df_payments_silver = (
    df_payments
    .withColumn("payment_type", initcap(col("payment_type")))
)
df_payments_silver.write.format("delta").mode("overwrite").saveAsTable("silver_payments")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 11, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import to_timestamp

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 12, Finished, Available, Finished, False)

In [11]:
df_reviews_silver = (
    df_reviews
    .filter(col("order_id").isNotNull())
    .filter(col("review_id").isNotNull())
    .withColumn("review_score", col("review_score").cast("integer"))
    .withColumn("review_comment_title", when(col("review_comment_title").isNull(), "No_Title"))
    .withColumn("review_comment_message", when(col("review_comment_message").isNull(), "No Comment")
        .otherwise(col("review_comment_message")))
    .withColumn("review_creation_date", to_timestamp(col("review_creation_date"),"yyyy-MM-dd HH:mm:ss"))
    .withColumn("review_answer_timestamp", to_timestamp(col("review_answer_timestamp"), "yyyy-MM-dd HH:mm:ss"))
)
df_reviews_silver.write.format("delta").mode("overwrite").saveAsTable("silver_reviews")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 13, Finished, Available, Finished, False)

aggregating and grouping the geo location coordinates to deduplicate data

In [12]:
from pyspark.sql.functions import avg, round
df_geolocation_silver =  (
    df_geolocation
    .withColumn("geolocation_city", initcap(col("geolocation_city")))
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(round(avg("geolocation_lat"), 6).alias("geolocation_lat"), round(avg("geolocation_lng"), 6).alias("geolocation_lng"))
)
df_geolocation_silver.write.format("delta").mode("overwrite").saveAsTable("silver_geolocation")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 14, Finished, Available, Finished, False)

adding price with freight value to get total cost of order

In [13]:
df_order_items_silver = (
    df_order_items
    .withColumn("total_item_value", round(col("price") + col("freight_value"),2))
)
df_order_items_silver.write.format("delta").mode("overwrite").saveAsTable("silver_order_items")

StatementMeta(, b8693feb-7c43-4c4e-a6b6-402e027d0b79, 15, Finished, Available, Finished, False)